In [1]:
import pandas as pd
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt

# All trade combiantions baseline 
SUMMARY_DIR = Path("./results_merged")
files = sorted(SUMMARY_DIR.glob("summary_*.csv"))
df = pd.concat(
    (pd.read_csv(f, parse_dates=["day"]) for f in files),
    ignore_index=True
)
df["day"] = pd.to_datetime(df["day"] , utc=True).dt.normalize()


ri_base = (
    df[df["combo_id"] == 0][["day", "profit"]]
    .rename(columns={"profit": "RI_base"})
)

myopic_base = (
    df.sort_values("da_profit", ascending=False)
    .groupby("day", as_index=False)
    .first()[["day", "profit"]]
    .rename(columns={"profit": "myopic_base"})
)


df = df.merge(ri_base, on="day", how="left")
df["uplift_ri_base_abs"] = df["profit"] - df["RI_base"]
df["uplift_ri_base_pct"] = df["uplift_ri_base_abs"] / df["RI_base"]

In [2]:
start_date = pd.Timestamp("2019-01-01", tz="UTC")
end_date = pd.Timestamp("2025-12-31", tz="UTC")

df = df.loc[(df["day"] >= start_date) & (df["day"] <= end_date)].copy()

#### Load drl observation space features

In [3]:
drl_obs_data = pd.read_csv("../data/simplified_data_jan_with_exaa_and_id_full/df_spot_train_2019-01-01_2025-09-30_with_features_utc.csv")
drl_obs_data["time"] = pd.to_datetime(drl_obs_data["time"], utc=True)
drl_obs_data.set_index("time", inplace=True)
drl_obs_data = drl_obs_data.resample('D').mean()
drl_obs_data.reset_index(inplace=True)
drl_obs_data["date"] = drl_obs_data["time"].dt.date
drl_obs_data

df["day"] = pd.to_datetime(df["day"].astype(str).str.slice(0, 10), errors="coerce")
df["day_utc"] = df["day"].dt.tz_localize("UTC")
df["date"] = df["day_utc"].dt.date
df

df_merged = df.merge(
    drl_obs_data,
    on="date",
    how="left"
)

In [4]:
df_merged.describe()

,day,combo_id,buy_hour,sell_hour,profit,da_profit,da_buy_price,da_sell_price,da_profit_share,RI_base,...,spread_id_full_da_qh_std,spread_id_full_da_qh_min,spread_id_full_da_qh_max,exaa_pf_daily_mean,exaa_pf_daily_std,exaa_pf_daily_min,exaa_pf_daily_max,exaa_pf_daily_spread,exaa_pf_daily_diff_sum,exaa_pf_daily_diff_max
count,747900,747900.000000,745407.000000,688068.000000,747900.000000,747900.000000,743314.000000,685975.000000,747900.000000,747900.000000,...,738600.000000,738600.000000,738600.000000,738600.000000,738600.000000,738600.000000,738600.000000,738600.000000,738600.000000,738600.000000
mean,2022-05-31 08:48:31.191335680,149.500000,8.615385,16.666667,180.751178,-10.682182,90.053147,99.923231,-0.084831,184.589212,...,18.182304,-30.242498,44.320203,95.246535,27.310009,52.825916,144.496384,91.670468,243.975470,32.466504
min,2019-01-01 00:00:00,0.000000,1.000000,2.000000,-164.210000,-936.280000,-500.000000,-500.000000,-17.620000,17.538780,...,2.049748,-666.830490,-29.265286,-11.700625,2.458386,-157.148333,8.163333,7.754167,29.085833,3.246667
25%,2020-09-15 00:00:00,74.750000,4.000000,13.000000,79.617660,-25.284000,35.030000,39.890000,-0.190000,84.947166,...,7.556806,-38.231138,11.851631,40.658455,9.879148,15.101667,57.715000,33.140833,89.616667,11.601250
50%,2022-05-31 00:00:00,149.500000,8.000000,18.000000,141.950158,-5.233100,69.750000,75.900000,-0.050000,145.283671,...,13.231110,-20.790562,24.563226,71.928021,21.563249,34.151250,118.784167,73.732083,200.626875,25.224583
75%,2024-02-13 00:00:00,224.250000,13.000000,21.000000,231.242731,9.407050,108.400000,122.470000,0.080000,234.392655,...,21.302582,-11.250312,47.925669,111.355278,38.590540,73.003750,175.181667,127.683333,329.366667,43.382500
max,2025-10-30 00:00:00,299.000000,23.000000,24.000000,2589.572650,697.850800,936.280000,936.280000,80.200000,2375.525044,...,511.818885,38.438722,2065.799809,671.490694,171.314541,562.578333,970.827500,765.774167,2187.145833,605.714583
std,NaN,86.602117,5.704772,5.527712,157.983589,57.888923,90.696571,98.905191,0.318663,157.542834,...,23.711049,35.555932,94.322838,87.730407,21.949936,69.688068,121.232708,74.136297,199.260252,31.473568


In [5]:
df1= df_merged.copy()

In [6]:
df_merged.columns

Index(['day', 'combo_id', 'kind', 'buy_hour', 'sell_hour', 'profit',
       'da_exec_time', 'da_profit', 'da_buy_price', 'da_sell_price',
       'da_profit_share', 'RI_base', 'uplift_ri_base_abs',
       'uplift_ri_base_pct', 'day_utc', 'date', 'time',
       'epex_spot_60min_de_lu_eur_per_mwh', 'exaa_15min_de_lu_eur_per_mwh',
       'load_forecast_d_minus_1_1000_total_de_lu_mw',
       'pv_forecast_d_minus_1_1000_de_lu_mw',
       'wind_offshore_forecast_d_minus_1_1000_de_lu_mw',
       'wind_onshore_forecast_d_minus_1_1000_de_lu_mw', 'date_month',
       'day_of_week', 'wind_forecast_daily_mean', 'wind_forecast_daily_std',
       'spread_id_full_da_h_mean', 'spread_id_full_da_h_std',
       'spread_id_full_da_h_min', 'spread_id_full_da_h_max',
       'spread_id_full_da_qh_mean', 'spread_id_full_da_qh_std',
       'spread_id_full_da_qh_min', 'spread_id_full_da_qh_max',
       'exaa_pf_daily_mean', 'exaa_pf_daily_std', 'exaa_pf_daily_min',
       'exaa_pf_daily_max', 'exaa_pf_daily_spr

#### Prepare input data

In [7]:
columns_to_keep = [ 'date','combo_id','buy_hour', 'sell_hour',
                   'profit',
       'load_forecast_d_minus_1_1000_total_de_lu_mw',
       'pv_forecast_d_minus_1_1000_de_lu_mw',
       'wind_offshore_forecast_d_minus_1_1000_de_lu_mw',
       'wind_onshore_forecast_d_minus_1_1000_de_lu_mw', 'date_month',
       'day_of_week', 'wind_forecast_daily_mean', 'wind_forecast_daily_std',
       'spread_id_full_da_h_mean', 'spread_id_full_da_h_std',
       'spread_id_full_da_h_min', 'spread_id_full_da_h_max',
       'spread_id_full_da_qh_mean', 'spread_id_full_da_qh_std',
       'spread_id_full_da_qh_min', 'spread_id_full_da_qh_max',
       'exaa_pf_daily_mean', 'exaa_pf_daily_std', 'exaa_pf_daily_min',
       'exaa_pf_daily_max', 'exaa_pf_daily_spread', 'exaa_pf_daily_diff_sum',
       'exaa_pf_daily_diff_max']

df = df_merged[columns_to_keep].copy()



# Clean up / enforce types
for c in ["buy_hour", "sell_hour"]:
    df[c] = df[c].fillna(0).astype(int)

# Add explicit time features (requested)
_dt = pd.to_datetime(df["date"])  # `date` is a python datetime.date
df["date_year"] = _dt.dt.year.astype(int)
df["date_month_num"] = _dt.dt.month.astype(int)

# Ensure `date_month` is numeric if you want to keep using it
if "date_month" in df.columns:
    df["date_month"] = pd.to_numeric(df["date_month"], errors="coerce")

#### Prepare model

In [8]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [9]:
# --- Optional manual override of train/test date ranges ---
# Set these to pd.Timestamp(..., tz="UTC") to activate a manual split.
# Leave them as None to use the automatic forward split in the next cell.

train_start_date = pd.Timestamp("2019-01-01", tz="UTC")
train_end_date =  pd.Timestamp("2021-03-31", tz="UTC")

test_start_date = pd.Timestamp("2021-04-01", tz="UTC")
test_end_date = pd.Timestamp("2021-09-30", tz="UTC")

print(
    "[manual split config]",
    dict(
        train_start_date=train_start_date,
        train_end_date=train_end_date,
        test_start_date=test_start_date,
        test_end_date=test_end_date,
    ),
)

[manual split config] {'train_start_date': Timestamp('2019-01-01 00:00:00+0000', tz='UTC'), 'train_end_date': Timestamp('2021-03-31 00:00:00+0000', tz='UTC'), 'test_start_date': Timestamp('2021-04-01 00:00:00+0000', tz='UTC'), 'test_end_date': Timestamp('2021-09-30 00:00:00+0000', tz='UTC')}


In [10]:
# Target
y_col = "profit"

# Group (per day)
group_col = "date"

# Excludes (target + group)
exclude = {
    y_col,
    group_col,
}

# Features
feature_cols = [c for c in df.columns if c not in exclude]
X = df[feature_cols]
y = df[y_col]

# Manual split (if configured in the previous cell)
manual_active = all(
    v is not None
    for v in [train_start_date, train_end_date, test_start_date, test_end_date]
)

if manual_active:
    _date_ts = pd.to_datetime(df[group_col]).dt.tz_localize("UTC")

    train_mask = (_date_ts >= train_start_date) & (_date_ts <= train_end_date)
    test_mask = (_date_ts >= test_start_date) & (_date_ts <= test_end_date)

    overlap = int((train_mask & test_mask).sum())
    if overlap > 0:
        raise ValueError(f"Train/test date ranges overlap in {overlap} rows. Fix the ranges.")

    split_kind = "manual"
else:
    # Forward (time-ordered) split by day (default)
    unique_days = pd.Series(df[group_col].unique()).sort_values()
    test_size = 0.2
    split_at = int(np.floor((1 - test_size) * len(unique_days)))
    train_days = set(unique_days.iloc[:split_at])
    test_days = set(unique_days.iloc[split_at:])

    train_mask = df[group_col].isin(train_days)
    test_mask = df[group_col].isin(test_days)

    split_kind = "auto_forward"

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

df_train = df.loc[train_mask].copy()
df_test = df.loc[test_mask].copy()

print("split:", split_kind)
print("n_days train/test:", df_train[group_col].nunique(), df_test[group_col].nunique())
print("rows train/test:", len(df_train), len(df_test))
if len(df_train) > 0:
    print("date range train:", df_train[group_col].min(), "->", df_train[group_col].max())
if len(df_test) > 0:
    print("date range test:", df_test[group_col].min(), "->", df_test[group_col].max())

split: manual
n_days train/test: 821 183
rows train/test: 246300 54900
date range train: 2019-01-01 -> 2021-03-31
date range test: 2021-04-01 -> 2021-09-30


In [11]:
import numpy as np
import pandas as pd

from lightgbm import LGBMRanker

# ---------- helpers: ranking metrics per day ----------

def add_relevance_rank(df_, y_col="profit"):
    # Higher profit -> higher relevance (0..n-1)
    # LightGBM ranking requires integer labels.
    df_ = df_.copy()
    df_["relevance"] = (
        df_.groupby("date")[y_col]
        .rank(method="average", ascending=True)
        .astype(int)
        - 1
    )
    return df_


def ndcg_at_k(df_, k=10, score_col="score"):
    # NDCG@k: compares your predicted ranking to the ideal ranking.
    # IMPORTANT: the classic gain (2**rel - 1) assumes small rel levels (e.g., 0..4).
    # Here we often have rel in 0..299, so we use *linear gain* to avoid overflow.
    out = []
    skipped = 0

    for day, g in df_.groupby("date"):
        if len(g) == 0 or "relevance" not in g.columns:
            skipped += 1
            continue

        g = g.sort_values(score_col, ascending=False)
        rel = g["relevance"].to_numpy()
        rel = rel[~pd.isna(rel)]
        if len(rel) == 0:
            skipped += 1
            continue

        rel_k = rel[: min(k, len(rel))].astype(float)
        denom = np.log2(np.arange(2, len(rel_k) + 2))

        # Linear-gain DCG
        dcg = float(np.sum(rel_k / denom))

        ideal = np.sort(rel)[::-1][: len(rel_k)].astype(float)
        idcg = float(np.sum(ideal / denom))

        if not np.isfinite(idcg) or idcg <= 0:
            skipped += 1
            continue

        out.append(dcg / idcg)

    if len(out) == 0:
        print(
            f"[ndcg_at_k] could not compute NDCG: days_total={df_['date'].nunique()} skipped={skipped}"
        )
        return float("nan")

    if skipped > 0:
        print(f"[ndcg_at_k] computed on {len(out)} days; skipped={skipped}")

    return float(np.mean(out))


def spearman_per_day(df_, y_col="profit", score_col="score"):
    vals = []
    for _, g in df_.groupby("date"):
        if g[y_col].nunique() <= 1:
            continue
        vals.append(g[[y_col, score_col]].corr(method="spearman").iloc[0, 1])
    return float(np.nanmean(vals))


def precision_at_k(df_, k=10, top_frac=0.1, y_col="profit", score_col="score"):
    vals = []
    for _, g in df_.groupby("date"):
        n = len(g)
        if n == 0:
            continue
        true_top_n = max(1, int(np.ceil(top_frac * n)))
        true_top = set(g.nlargest(true_top_n, y_col).index)
        pred_top = set(g.nlargest(min(k, n), score_col).index)
        vals.append(len(true_top & pred_top) / max(1, len(pred_top)))
    return float(np.mean(vals))


def top1_regret(df_, y_col="profit", score_col="score"):
    regs = []
    for _, g in df_.groupby("date"):
        oracle = g[y_col].max()
        chosen = g.loc[g[score_col].idxmax(), y_col]
        regs.append(oracle - chosen)
    return float(np.mean(regs)), float(np.median(regs))


# ---------- build integer relevance labels for LightGBM ----------

df_train_rel = add_relevance_rank(df_train, y_col="profit")
df_test_rel = add_relevance_rank(df_test, y_col="profit")

y_train_rel = df_train_rel["relevance"].astype(int)
y_test_rel = df_test_rel["relevance"].astype(int)

# Group sizes must align with the row order of X_train / X_test.
# Since X_train/X_test were created via boolean masks on `df` (preserving row order),
# we compute sizes in that same order by grouping the corresponding df slices.
train_group_sizes = df_train.groupby("date", sort=False).size().to_numpy()
test_group_sizes = df_test.groupby("date", sort=False).size().to_numpy()

# ---------- experiments (ablations) ----------

max_rel = int(max(y_train_rel.max(), y_test_rel.max()))
label_gain = list(range(max_rel + 1))  # gain per relevance level


def fit_ranker(X_tr, y_tr, X_va, y_va, train_groups, val_groups, *, seed=42):
    model = LGBMRanker(
        objective="lambdarank",
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        label_gain=label_gain,
    )
    model.fit(
        X_tr,
        y_tr,
        group=train_groups,
        eval_set=[(X_va, y_va)],
        eval_group=[val_groups],
        eval_at=[5, 10, 20],
    )
    return model


def evaluate(name, df_test_base, scores, *, top_frac=0.10):
    df_sc = df_test_base.copy()
    df_sc["score"] = scores

    row = {
        "model": name,
        "NDCG@5": ndcg_at_k(df_sc, k=5, score_col="score"),
        "NDCG@10": ndcg_at_k(df_sc, k=10, score_col="score"),
        "NDCG@20": ndcg_at_k(df_sc, k=20, score_col="score"),
        "Spearman": spearman_per_day(df_sc, score_col="score"),
        "Prec@10(top10%)": precision_at_k(df_sc, k=10, top_frac=top_frac, score_col="score"),
        "Prec@30(top10%)": precision_at_k(df_sc, k=30, top_frac=top_frac, score_col="score"),
    }

    mean_reg, med_reg = top1_regret(df_sc, score_col="score")
    row["Regret_mean"] = mean_reg
    row["Regret_median"] = med_reg

    chosen = df_sc.loc[df_sc.groupby("date")["score"].idxmax()]
    oracle = df_sc.loc[df_sc.groupby("date")["profit"].idxmax()]
    row["MeanProfit_chosen"] = float(chosen["profit"].mean())
    row["MeanProfit_oracle"] = float(oracle["profit"].mean())

    return row


# Feature sets
combo_features = ["buy_hour", "sell_hour"]
if "combo_id" in X.columns:
    combo_features = ["combo_id"] + combo_features

X_train_combo = X_train[combo_features]
X_test_combo = X_test[combo_features]

# Train models
ranker_full = fit_ranker(
    X_train,
    y_train_rel,
    X_test,
    y_test_rel,
    train_group_sizes,
    test_group_sizes,
)

ranker_combo = fit_ranker(
    X_train_combo,
    y_train_rel,
    X_test_combo,
    y_test_rel,
    train_group_sizes,
    test_group_sizes,
)

# Score + collect results
rows = []
rows.append(evaluate("LightGBM ranker (forecast + combo)", df_test_rel, ranker_full.predict(X_test)))
rows.append(evaluate("LightGBM ranker (combo only)", df_test_rel, ranker_combo.predict(X_test_combo)))

# Baseline: random
rng = np.random.default_rng(42)
rows.append(evaluate("Baseline: random", df_test_rel, rng.standard_normal(len(df_test_rel))))

# Baseline: historical mean per combo_id (train only)
combo_mean = df_train.groupby("combo_id")["profit"].mean()
df_hist = df_test_rel.copy()
df_hist["score"] = df_hist["combo_id"].map(combo_mean).fillna(combo_mean.mean())
rows.append(evaluate("Baseline: historical mean per combo_id", df_test_rel, df_hist["score"].to_numpy()))

results = pd.DataFrame(rows)

# Pretty print
pd.set_option("display.max_columns", 50)
display(
    results.assign(
        **{
            "NDCG@5": results["NDCG@5"].round(4),
            "NDCG@10": results["NDCG@10"].round(4),
            "NDCG@20": results["NDCG@20"].round(4),
            "Spearman": results["Spearman"].round(4),
            "Prec@10(top10%)": results["Prec@10(top10%)"].round(4),
            "Prec@30(top10%)": results["Prec@30(top10%)"].round(4),
            "Regret_mean": results["Regret_mean"].round(3),
            "Regret_median": results["Regret_median"].round(3),
            "MeanProfit_chosen": results["MeanProfit_chosen"].round(3),
            "MeanProfit_oracle": results["MeanProfit_oracle"].round(3),
        }
    )
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004526 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5722
[LightGBM] [Info] Number of data points in the train set: 246300, number of used features: 28
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001257 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 298
[LightGBM] [Info] Number of data points in the train set: 246300, number of used features: 3


,model,NDCG@5,NDCG@10,NDCG@20,Spearman,Prec@10(top10%),Prec@30(top10%),Regret_mean,Regret_median,MeanProfit_chosen,MeanProfit_oracle
0,LightGBM ranker (forecast + combo),0.5495,0.5579,0.5611,0.0784,0.1175,0.1137,27.175,18.443,142.447,169.621
1,LightGBM ranker (combo only),0.5460,0.5621,0.5697,0.1276,0.1290,0.1308,26.308,20.935,143.313,169.621
2,Baseline: random,0.5062,0.5121,0.5166,0.0097,0.0962,0.1058,27.494,21.136,142.127,169.621
3,Baseline: historical mean per combo_id,0.5532,0.5557,0.5631,0.1177,0.1153,0.1129,26.308,20.935,143.313,169.621


In [12]:
# results.to_csv("analysis_prediction_test_results.csv", index=False)

#### Rolling Cross Validation (2y train, 6m test, 6m step)

In [13]:
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Rolling CV setup: 2 years train, 6 months test, 6 months step
# ------------------------------------------------------------

def build_rolling_splits(
    df_,
    date_col="date",
    train_months=24,
    test_months=6,
    step_months=6,
):
    date_ts = pd.to_datetime(df_[date_col])
    min_date = date_ts.min().normalize()
    max_date = date_ts.max().normalize()

    splits = []
    fold_id = 0
    train_start = min_date

    while True:
        train_end_exclusive = train_start + pd.DateOffset(months=train_months)
        test_end_exclusive = train_end_exclusive + pd.DateOffset(months=test_months)

        if test_end_exclusive > (max_date + pd.Timedelta(days=1)):
            break

        train_mask = (date_ts >= train_start) & (date_ts < train_end_exclusive)
        test_mask = (date_ts >= train_end_exclusive) & (date_ts < test_end_exclusive)

        if train_mask.any() and test_mask.any():
            splits.append(
                {
                    "fold": fold_id,
                    "train_start": train_start,
                    "train_end": train_end_exclusive - pd.Timedelta(days=1),
                    "test_start": train_end_exclusive,
                    "test_end": test_end_exclusive - pd.Timedelta(days=1),
                    "train_mask": train_mask,
                    "test_mask": test_mask,
                }
            )
            fold_id += 1

        train_start = train_start + pd.DateOffset(months=step_months)

    return splits


ORACLE_MODE = "top30_mean"  # options: "max", "p90", "top30_mean"


def compute_daily_oracle(df_, mode="top30_mean", y_col="profit"):
    g = df_.groupby("date")[y_col]
    if mode == "max":
        return g.max()
    if mode == "p90":
        return g.quantile(0.90)
    if mode == "top30_mean":
        return g.apply(lambda s: s.nlargest(min(30, len(s))).mean())
    raise ValueError(f"Unsupported ORACLE_MODE: {mode}")


def evaluate_from_scores(name, df_test_base, scores, *, top_frac=0.10, oracle_mode=ORACLE_MODE):
    df_sc = df_test_base.copy()
    df_sc["score"] = scores

    row = {
        "model": name,
        "NDCG@30": ndcg_at_k(df_sc, k=30, score_col="score"),
        "Spearman": spearman_per_day(df_sc, score_col="score"),
        "Prec@30(top10%)": precision_at_k(df_sc, k=30, top_frac=top_frac, score_col="score"),
    }

    chosen = df_sc.loc[df_sc.groupby("date")["score"].idxmax()].set_index("date")
    oracle_daily = compute_daily_oracle(df_sc, mode=oracle_mode, y_col="profit")

    row["MeanProfit_chosen"] = float(chosen["profit"].mean())
    row["MeanProfit_oracle"] = float(oracle_daily.mean())
    row["Regret_mean"] = float((oracle_daily - chosen["profit"]).mean())
    row["n_test_days"] = int(df_sc["date"].nunique())

    return row, df_sc


splits = build_rolling_splits(df, date_col="date", train_months=24, test_months=6, step_months=6)
print(f"n_folds: {len(splits)}")

combo_features = ["buy_hour", "sell_hour"]
if "combo_id" in X.columns:
    combo_features = ["combo_id"] + combo_features

fold_rows = []
oof_frames = []

for s in splits:
    fold = s["fold"]

    df_train_fold = df.loc[s["train_mask"]].copy()
    df_test_fold = df.loc[s["test_mask"]].copy()

    X_train_fold = X.loc[s["train_mask"]]
    X_test_fold = X.loc[s["test_mask"]]

    df_train_rel = add_relevance_rank(df_train_fold, y_col="profit")
    df_test_rel = add_relevance_rank(df_test_fold, y_col="profit")

    y_train_rel = df_train_rel["relevance"].astype(int)
    y_test_rel = df_test_rel["relevance"].astype(int)

    train_group_sizes = df_train_fold.groupby("date", sort=False).size().to_numpy()
    test_group_sizes = df_test_fold.groupby("date", sort=False).size().to_numpy()

    max_rel_fold = int(max(y_train_rel.max(), y_test_rel.max()))
    label_gain_fold = list(range(max_rel_fold + 1))

    def fit_ranker_fold(X_tr, y_tr, X_va, y_va, train_groups, val_groups, *, seed=42):
        model = LGBMRanker(
            objective="lambdarank",
            n_estimators=2000,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=5,
            min_child_samples=400,
            feature_fraction=0.65,
            bagging_fraction=0.7,
            bagging_freq=1,
            lambda_l1=5.0,
            lambda_l2=20.0,
            min_split_gain=0.1,
            random_state=seed,
            label_gain=label_gain_fold,
        )
        model.fit(
            X_tr,
            y_tr,
            group=train_groups,
            eval_set=[(X_va, y_va)],
            eval_group=[val_groups],
            eval_at=[30],
        )
        return model

    ranker_full = fit_ranker_fold(
        X_train_fold,
        y_train_rel,
        X_test_fold,
        y_test_rel,
        train_group_sizes,
        test_group_sizes,
    )

    X_train_combo_fold = X_train_fold[combo_features]
    X_test_combo_fold = X_test_fold[combo_features]

    ranker_combo = fit_ranker_fold(
        X_train_combo_fold,
        y_train_rel,
        X_test_combo_fold,
        y_test_rel,
        train_group_sizes,
        test_group_sizes,
    )

    model_scores = {
        "LightGBM ranker (forecast + combo)": ranker_full.predict(X_test_fold),
        "LightGBM ranker (combo only)": ranker_combo.predict(X_test_combo_fold),
    }

    rng = np.random.default_rng(42 + fold)
    model_scores["Baseline: random"] = rng.standard_normal(len(df_test_rel))

    combo_mean = df_train_fold.groupby("combo_id")["profit"].mean()
    hist_score = df_test_rel["combo_id"].map(combo_mean).fillna(combo_mean.mean()).to_numpy()
    model_scores["Baseline: historical mean per combo_id"] = hist_score

    for model_name, score_vec in model_scores.items():
        row, df_sc = evaluate_from_scores(model_name, df_test_rel, score_vec)
        row.update(
            {
                "fold": fold,
                "train_start": s["train_start"].date(),
                "train_end": s["train_end"].date(),
                "test_start": s["test_start"].date(),
                "test_end": s["test_end"].date(),
            }
        )
        fold_rows.append(row)

        oof_tmp = df_sc[["date", "profit", "score"]].copy()
        oof_tmp["model"] = model_name
        oof_tmp["fold"] = fold
        oof_frames.append(oof_tmp)

fold_results = pd.DataFrame(fold_rows)

metric_cols = [
    "NDCG@30",
    "Spearman",
    "Prec@30(top10%)",
    "Regret_mean",
    "MeanProfit_chosen",
    "MeanProfit_oracle",
]

agg_mean = fold_results.groupby("model", as_index=False)[metric_cols].mean()
agg_std = fold_results.groupby("model", as_index=False)[metric_cols].std().add_suffix("_std")
agg_std = agg_std.rename(columns={"model_std": "model"})

agg_results = agg_mean.merge(agg_std, on="model", how="left")

# Optional: pooled out-of-fold evaluation over all test windows combined
oof_all = pd.concat(oof_frames, ignore_index=True)
pooled_rows = []
for model_name, g in oof_all.groupby("model"):
    pooled_eval = {
        "model": model_name,
        "NDCG@30_pooled": ndcg_at_k(g, k=30, score_col="score"),
        "Spearman_pooled": spearman_per_day(g, score_col="score"),
        "Prec@30(top10%)_pooled": precision_at_k(g, k=30, top_frac=0.10, score_col="score"),
    }

    chosen = g.loc[g.groupby("date")["score"].idxmax()].set_index("date")
    oracle_daily = compute_daily_oracle(g, mode=ORACLE_MODE, y_col="profit")

    pooled_eval["MeanProfit_chosen_pooled"] = float(chosen["profit"].mean())
    pooled_eval["MeanProfit_oracle_pooled"] = float(oracle_daily.mean())
    pooled_eval["Regret_mean_pooled"] = float((oracle_daily - chosen["profit"]).mean())
    pooled_eval["n_oof_days"] = int(g["date"].nunique())

    pooled_rows.append(pooled_eval)

pooled_results = pd.DataFrame(pooled_rows)

pd.set_option("display.max_columns", 200)

print("\nFold-wise results (first rows):")
display(fold_results.head())

print("\nAggregated over folds (mean +/- std):")
display(agg_results.sort_values("NDCG@30", ascending=False))

print("\nPooled OOF metrics across all folds:")
display(pooled_results.sort_values("NDCG@30_pooled", ascending=False))

n_folds: 9
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] lambda_l1 is set=5.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=5.0
[LightGBM] [Warning] lambda_l2 is set=20.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=20.0
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] feature_fraction is set=0.65, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.65
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] lambda_l1 is set=5.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=5.0
[LightGBM] [Warning] lambda_l2 is set=20.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=20.0
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction

,model,NDCG@30,Spearman,Prec@30(top10%),MeanProfit_chosen,MeanProfit_oracle,Regret_mean,n_test_days,fold,train_start,train_end,test_start,test_end
0,LightGBM ranker (forecast + combo),0.558568,0.111487,0.110313,114.515029,129.548949,15.033920,181,0,2019-01-01,2020-12-31,2021-01-01,2021-06-30
1,LightGBM ranker (combo only),0.551525,0.113943,0.111418,113.546504,129.548949,16.002444,181,0,2019-01-01,2020-12-31,2021-01-01,2021-06-30
2,Baseline: random,0.516642,-0.003580,0.095028,111.772090,129.548949,17.776859,181,0,2019-01-01,2020-12-31,2021-01-01,2021-06-30
3,Baseline: historical mean per combo_id,0.533875,0.105429,0.093554,113.546504,129.548949,16.002444,181,0,2019-01-01,2020-12-31,2021-01-01,2021-06-30
4,LightGBM ranker (forecast + combo),0.607614,0.202054,0.135870,224.728455,244.816869,20.088413,184,1,2019-07-01,2021-06-30,2021-07-01,2021-12-31



Aggregated over folds (mean +/- std):


,model,NDCG@30,Spearman,Prec@30(top10%),Regret_mean,MeanProfit_chosen,MeanProfit_oracle,NDCG@30_std,Spearman_std,Prec@30(top10%)_std,Regret_mean_std,MeanProfit_chosen_std,MeanProfit_oracle_std
3,LightGBM ranker (forecast + combo),0.584114,0.133354,0.128956,21.400043,230.963636,252.363679,0.019590,0.045582,0.010130,5.832303,87.398398,92.729017
2,LightGBM ranker (combo only),0.581989,0.122137,0.127555,21.029629,231.334051,252.363679,0.022357,0.046964,0.011516,5.700805,87.794228,92.729017
0,Baseline: historical mean per combo_id,0.571091,0.116976,0.119400,21.444409,230.919270,252.363679,0.019135,0.041832,0.012321,5.461441,87.935646,92.729017
1,Baseline: random,0.518944,-0.000410,0.098778,24.334257,228.029422,252.363679,0.004459,0.003653,0.005027,6.936187,86.089139,92.729017



Pooled OOF metrics across all folds:


,model,NDCG@30_pooled,Spearman_pooled,Prec@30(top10%)_pooled,MeanProfit_chosen_pooled,MeanProfit_oracle_pooled,Regret_mean_pooled,n_oof_days
0,Baseline: historical mean per combo_id,NaN,0.117070,0.119451,231.193677,252.644428,21.450751,1640
1,Baseline: random,NaN,-0.000397,0.098780,228.289154,252.644428,24.355274,1640
2,LightGBM ranker (combo only),NaN,0.122235,0.127602,231.602247,252.644428,21.042180,1640
3,LightGBM ranker (forecast + combo),NaN,0.133436,0.128984,231.226872,252.644428,21.417555,1640
